In [1]:
import sys
print(sys.executable)

C:\Users\hhdor\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe


In [7]:
import sys
!"{sys.executable}" -m pip install -U docling

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   - -------------------------------------- 0.8/16.1 MB 5.6 MB/s eta 0:00:03
   ---- ----------------------------------- 1.8/16.1 MB 4.8 MB/s eta 0:00:03
   ------ --------------------------------- 2.6/16.1 MB 4.4 MB/s eta 0:00:04
   ------- ----------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.


In [ ]:
from langchain_ollama import ChatOllama
from langchain_docling import ...
from langchain_huggingface import ...
from langchain_core.tools import ...


llm = ChatOllama(
    model="ministral-3",
    base_url="http://127.0.0.1:11434",
    
)


agent = create_agent(
    model = llm,
    tools=[...],
    system_prompt= ("system",
     "Tu es un assistant d'analyse documentaire (DDRM / risques). "
     "RÈGLE ABSOLUE: tu dois répondre UNIQUEMENT à partir du CONTEXTE fourni par l'outil. "
     "Si le contexte ne contient pas l'information, réponds exactement: "
     "'Information non trouvée dans les documents fournis.' "
     "Réponds en français, clair, structuré (étapes, listes). "
     "Ajoute des citations [source:...] sur les points importants.")
    
)

Bonjour ! 😊 Comment puis-je vous aider aujourd'hui ?


SyntaxError: invalid syntax (761061002.py, line 1)

In [ ]:
from langchain_ollama import ChatOllama
from langchain_docling import ...
from langchain_huggingface import ...
from langchain_core.tools import ...

agent = create_agent(
    model = llm,
    tools=[....],
    system_prompt= "",
    
)

SyntaxError: invalid syntax (2922945540.py, line 3)

In [3]:
import os
from pathlib import Path
from typing import List, Optional

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool

# Docling (PDF -> Markdown)
from docling.document_converter import DocumentConverter

# (Optionnel) Mémoire conversationnelle simple
from langchain_core.messages import SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate


# -----------------------------
# 0) CONFIG
# -----------------------------
OLLAMA_URL = "http://127.0.0.1:11434"
LLM_MODEL = "ministral-3"
EMBED_MODEL = "nomic-embed-text"   # embeddings (local via ollama)
INDEX_DIR = Path("index_ddrm_faiss")

# Tu mets tes PDFs ici
DOCS_DIR = Path("docs")  # ex: docs/ddrm_vienne.pdf



In [4]:
# -----------------------------
# 1) INJECTION DOCS -> Markdown -> Chunks
# -----------------------------
def pdf_to_markdown(pdf_path: Path) -> str:
    converter = DocumentConverter()
    result = converter.convert(str(pdf_path))
    return result.document.export_to_markdown()


def load_documents_from_folder(folder: Path) -> List[Document]:
    docs: List[Document] = []
    for p in folder.rglob("*"):
        if p.suffix.lower() == ".pdf":
            md = pdf_to_markdown(p)
            docs.append(Document(page_content=md, metadata={"source": str(p), "type": "pdf"}))
        elif p.suffix.lower() in [".md", ".markdown"]:
            md = p.read_text(encoding="utf-8")
            docs.append(Document(page_content=md, metadata={"source": str(p), "type": "md"}))
    return docs



In [5]:
def chunk_documents(docs: List[Document]) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=4000,
        chunk_overlap=400,
        separators=["\n### ", "\n## ", "\n# ", "\n\n", "\n", " "],
    )
    return splitter.split_documents(docs)


In [ ]:
# -----------------------------
# 2) VECTOR STORE (FAISS)
# -----------------------------
def build_or_load_vectorstore() -> FAISS:
    emb = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_URL)

    if INDEX_DIR.exists():
        # FAISS local load
        return FAISS.load_local(str(INDEX_DIR), emb, allow_dangerous_deserialization=True)

    # Build
    raw_docs = load_documents_from_folder(DOCS_DIR)
    if not raw_docs:
        raise FileNotFoundError(f"Aucun document trouvé dans {DOCS_DIR.resolve()} (.pdf/.md)")

    chunks = chunk_documents(raw_docs)
    vs = FAISS.from_documents(chunks, emb)
    vs.save_local(str(INDEX_DIR))
    return vs


VECTORSTORE = build_or_load_vectorstore()
RETRIEVER = VECTORSTORE.as_retriever(search_kwargs={"k": 6})



[INFO] 2026-01-22 08:48:30,852 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-01-22 08:48:31,004 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-01-22 08:48:31,024 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.5.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-01-22 08:48:40,277 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-01-22 08:48:58,264 [RapidOCR] download_file.py:95: Successfully saved to: C:\Users\hhdor\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-01-22 08:48:58,278 [RapidOCR] main.py:50: Using C:\Users\hhdor\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-01-22 08:48:59,261 [RapidOCR] base.py:22:

In [ ]:


def format_context(docs: List[Document]) -> str:
    # On met des "citations" simples par source
    blocks = []
    for d in docs:
        src = d.metadata.get("source", "unknown")
        blocks.append(f"[source:{src}]\n{d.page_content}")
    return "\n\n---\n\n".join(blocks)


# -----------------------------
# 3) TOOL: search_docs(query)
# -----------------------------
@tool
def search_docs(question: str) -> str:
    """
    Recherche les passages pertinents dans les documents indexés.
    Retourne le contexte avec des sources.
    """
    docs = RETRIEVER.get_relevant_documents(question)

    # Garde-fou: si aucun doc pertinent, on renvoie vide -> le LLM devra refuser
    if not docs:
        return ""

    return format_context(docs)


# -----------------------------
# 4) LLM + PROMPT (répond uniquement depuis contexte)
# -----------------------------
llm = ChatOllama(model=LLM_MODEL, base_url=OLLAMA_URL, temperature=0.2)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Tu es un assistant d'analyse documentaire (DDRM / risques). "
     "RÈGLE ABSOLUE: tu dois répondre UNIQUEMENT à partir du CONTEXTE fourni par l'outil. "
     "Si le contexte ne contient pas l'information, réponds exactement: "
     "'Information non trouvée dans les documents fournis.' "
     "Réponds en français, clair, structuré (étapes, listes). "
     "Ajoute des citations [source:...] sur les points importants."),
    ("human",
     "QUESTION: {question}\n\nCONTEXTE:\n{context}")
])


def answer_from_docs(question: str) -> str:
    context = search_docs.invoke({"question": question})
    # context est une string
    res = (prompt | llm).invoke({"question": question, "context": context})
    return res.content


# -----------------------------
# 5) MEMORY (historique)
# -----------------------------
history = InMemoryChatMessageHistory()

def get_session_history(_session_id: str):
    return history

# On wrap la chaîne finale avec mémoire (optionnel)
chain_with_memory = RunnableWithMessageHistory(
    runnable=(prompt | llm),
    get_session_history=get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)


def chat(question: str, session_id: str = "default") -> str:
    context = search_docs.invoke({"question": question})
    # Si pas de contexte -> on laisse le prompt refuser
    out = chain_with_memory.invoke(
        {"question": question, "context": context},
        config={"configurable": {"session_id": session_id}}
    )
    return out.content


# -----------------------------
# 6) UTILISATION
# -----------------------------
print(chat("Que faire en cas d'inondation ?"))


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


KeyboardInterrupt: 